In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/processed/casas.csv')

In [3]:
df.head()

,tamanho,ano,garagem,preco
0,159.0,2003,2,208500
1,117.0,1976,2,181500
2,166.0,2001,2,223500
3,160.0,1915,3,140000
4,204.0,2000,3,250000


In [4]:
X = df.drop('preco', axis=1)
y = df['preco']

In [5]:
X.head()

,tamanho,ano,garagem
0,159.0,2003,2
1,117.0,1976,2
2,166.0,2001,2
3,160.0,1915,3
4,204.0,2000,3


In [6]:
X_trian, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [7]:
import mlflow
mlflow.set_tracking_uri("file:///mnt/c/Users/Paulo/Documents/mlflow-alura/mlruns")

In [8]:
mlflow.set_experiment('house-prices-eda')

2025/01/13 19:15:08 INFO mlflow.tracking.fluent: Experiment with name 'house-prices-eda' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///mnt/c/Users/Paulo/Documents/mlflow-alura/mlruns/826371684564438656', creation_time=1736806508143, experiment_id='826371684564438656', last_update_time=1736806508143, lifecycle_stage='active', name='house-prices-eda', tags={}>

# Linear Regression

In [9]:
mlflow.start_run()

<ActiveRun: >

In [10]:
from sklearn.linear_model import LinearRegression
import math

lr = LinearRegression()
lr.fit(X_trian, y_train)
lr_predict = lr.predict(X_test)

In [11]:
mlflow.sklearn.log_model(lr,'lr')

2025/01/13 19:15:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [14]:
from sklearn.metrics import mean_squared_error, r2_score
import math

In [15]:
mse = mean_squared_error(y_test, lr.predict(X_test))
rmse = math.sqrt(mse)
r2 = r2_score(y_test, lr_predict)
mlflow.log_metric('mse',mse)
mlflow.log_metric('rmse',rmse)
mlflow.log_metric('r2',r2)

In [16]:
mlflow.end_run()

In [19]:
from xgboost import XGBRFRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression

with mlflow.start_run():
    # Generate synthetic data
    X, y = make_regression(n_samples=100, n_features=20, noise=0.1, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Fit the model
    xgb = XGBRFRegressor(random_state=42)
    xgb.fit(X_train, y_train)
    mlflow.xgboost.log_model(xgb,'xgboost')
    xgb_predicted = xgb.predict(X_test)
    mse = mean_squared_error(y_test, xgb_predicted)
    rmse = math.sqrt(mse)
    r2 = r2_score(y_test, xgb_predicted)
    mlflow.log_metric('mse',mse)
    mlflow.log_metric('rmse',rmse)
    mlflow.log_metric('r2',r2)

/home/pog/miniconda3/envs/mlflow/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [19:21:28] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/01/13 19:21:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [20]:
mlflow.get_experiment_by_name('house-prices-eda')

<Experiment: artifact_location='file:///mnt/c/Users/Paulo/Documents/mlflow-alura/mlruns/826371684564438656', creation_time=1736806508143, experiment_id='826371684564438656', last_update_time=1736806508143, lifecycle_stage='active', name='house-prices-eda', tags={}>